In [5]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.memory import ConversationBufferMemory

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.1,
    streaming=True
)

memory = ConversationBufferMemory(return_messages=True)

def ejecutar_con_streaming_y_memoria(prompt_content):
    history = memory.load_memory_variables({})["history"]
    current_message = HumanMessage(content=prompt_content)
    full_response = ""
    
    for chunk in llm.stream(history + [current_message]):
        content = chunk.content
        print(content, end="", flush=True)
        full_response += content
    
    memory.save_context({"input": prompt_content}, {"output": full_response})
    return full_response

def modulo_few_shot():
    prompt = r'''Genera expresiones regulares con explicación:
    
Tarea: validar email básico
Regex: ^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$
Explicación: Valida formato básico de email.
    
Tarea: validar números chilenos (+56 9 XXXX XXXX)
Regex:'''
    ejecutar_con_streaming_y_memoria(prompt)

def modulo_debugging_cot():
    codigo_con_error = """
def calcular_promedio(numeros):
    total = 0
    for n in numeros:
        total += n
    return total / len(numeros)
    """
    
    prompt_base = """Analiza este código de producción paso a paso:

CÓDIGO:
{}

PASOS:
1. Comprensión del objetivo.
2. Análisis de flujo.
3. Identificación de fallos (ej. división por cero).
4. Solución corregida.

Análisis:"""
    
    prompt_final = prompt_base.format(codigo_con_error)
    ejecutar_con_streaming_y_memoria(prompt_final)

if __name__ == "__main__":
    modulo_few_shot()
    modulo_debugging_cot()

ModuleNotFoundError: No module named 'langchain_openai'